# Train and Evaluate a Cross-Generator Run (VM workbook)

**Cross-Generator Generalization in Deepfake Detection (IE7374)**

Run this after `00_setup_and_preprocess.ipynb` has produced the crops cache
(`data/processed/` + `data/manifests/crops.parquet`) on this VM.

It builds the leave-one-manipulation-out splits, trains one detector on 3 of the 4
FF++ methods, tests it on the held-out 4th (plus the SimSwap set when it exists),
and prints the seen-vs-unseen gap. Pick your run with the two variables in step 3.

The eight runs are 2 backbones (EfficientNet, XceptionNet) times 4 held-out methods.
Everyone takes one or two so the eight are shared; each writes a results JSON that
combines into the transfer matrix.

## 1. Confirm the crops cache is present
If this fails, run `00_setup_and_preprocess.ipynb` first.

In [ ]:
import pandas as pd, os
assert os.path.exists("data/manifests/crops.parquet"), "run 00_setup_and_preprocess.ipynb first"
m = pd.read_parquet("data/manifests/crops.parquet")
print("crops:", len(m))
print(m.groupby("method").size())

## 2. Build the leave-one-manipulation-out splits
Writes one identity-disjoint fold per held-out method into `data/splits/`.
For each fold, train+val are 3 methods plus real, test is the held-out method plus real.

In [ ]:
!python data/make_splits.py --manifest data/manifests/crops.parquet --out data/splits
!ls -la data/splits

## 3. Pick your run
All eight runs have a committed config in `configs/` (2 backbones x 4 held-out
methods). Set `RUN` to yours; the cell lists the options and validates your choice.

Optional **VM-only** overrides adapt to your GPU without editing the committed config:
set `AMP_OVERRIDE` / `BATCH_OVERRIDE` (e.g. `fp16` on P100 / V100, a smaller batch if
VRAM is tight). These change only how the run executes, not the science (seed, data,
split, lr stay from the committed config). When set, the notebook writes a gitignored
`configs/_local/` copy and trains from that, so the committed config stays authoritative.

In [ ]:
import glob, os, yaml

runs = {os.path.splitext(os.path.basename(f))[0]: f
        for f in sorted(glob.glob("configs/*.yaml")) if "preprocess" not in f}
print("available runs:")
for name in runs:
    print("   ", name)

# ADJUST 1: your run (name from the list above)
RUN = "xception_holdout-deepfakes"

# ADJUST 2 (optional, GPU hardware only): leave None to use the committed values
AMP_OVERRIDE = None      # e.g. "fp16" on P100 / V100
BATCH_OVERRIDE = None    # e.g. 16 if VRAM is tight

assert RUN in runs, f"no config for {RUN}. Pick from: {list(runs)}"
cfg = yaml.safe_load(open(runs[RUN]))
if AMP_OVERRIDE is not None:
    cfg["amp"] = AMP_OVERRIDE
if BATCH_OVERRIDE is not None:
    cfg["batch_size"] = BATCH_OVERRIDE
run_name = cfg["run_name"]

if AMP_OVERRIDE is not None or BATCH_OVERRIDE is not None:
    os.makedirs("configs/_local", exist_ok=True)
    cfg_path = f"configs/_local/{run_name}.yaml"
    yaml.safe_dump(cfg, open(cfg_path, "w"), sort_keys=False)
    print("\nusing machine-local override:", cfg_path)
else:
    cfg_path = runs[RUN]
    print("\nusing committed config:", cfg_path)

assert os.path.exists(cfg["manifest"]), "crops.parquet missing: run 00_setup_and_preprocess.ipynb"
assert os.path.exists(cfg["split"]), f"{cfg['split']} missing: run the splits cell (step 2)"
print(yaml.safe_dump(cfg, sort_keys=False))

## 4. Train
Fine-tunes the ImageNet-pretrained backbone on the 3 training methods plus real.
Saves a checkpoint under `checkpoints/<run_name>/`. Uses the GPU if one is visible
(check step 3 of the setup notebook). This is the long cell.

In [ ]:
!python experiments/train.py --config {cfg_path}

## 5. Evaluate
Scores the trained model on each method: the 3 seen methods (in-distribution) and
the held-out method (unseen), plus the SimSwap set if its split exists. Writes
`experiments/results/<run_name>.json` in the shared results schema.

In [ ]:
!python experiments/evaluate.py --config {cfg_path}

## 6. Read the result (the seen-vs-unseen gap)
The headline number is the drop from seen methods to the held-out (unseen) one.

In [ ]:
import json, pandas as pd
r = json.load(open(f"experiments/results/{run_name}.json"))
df = pd.DataFrame(r["results"])
print(f"run: {r['run_name']}  backbone: {r['backbone']}  held-out: {r['held_out_method']}  level: {r['level']}")
print(df[["tested_on", "seen", "auc", "acc", "f1"]].to_string(index=False))
seen = df[df.seen == True]["auc"].mean()
unseen = df[df.seen == False]["auc"].mean()
print(f"\nmean seen AUC:   {seen:.4f}")
print(f"mean unseen AUC: {unseen:.4f}")
print(f"generalization gap (seen - unseen): {seen - unseen:.4f}")

## 7. Share your result with the team
Commit and push just the small results JSON. Checkpoints and crops are gitignored,
so they never get pushed. This JSON is what feeds the transfer matrix in
`02_transfer_matrix.ipynb`, so without this step your run stays on your VM.

First time on a VM: make sure git is configured (`git config user.name` / `user.email`)
and that you can push (you are a repo collaborator; cloning over HTTPS caches your token).

In [ ]:
run_json = f"experiments/results/{run_name}.json"
assert os.path.exists(run_json), "no results JSON yet; run steps 4 and 5 first"
!git add {run_json}
!git commit -m "results: {run_name}"
!git pull --rebase
!git push

## Next steps
- Run the other folds / backbone (change `RUN` in step 3) so all eight results JSONs
  exist, then assemble them with `02_transfer_matrix.ipynb`.
- The held-out method is never opened during training, so its column is a clean
  unseen-generator measurement.
- Optional: log to a shared Weights and Biases project so the runs compare in one
  dashboard (see `docs/INTERFACES.md`, Contract 5).
- The `SimSwap` column stays empty until the self-generated set and its split exist.